# 03 协议的线级传输：A2A over HTTP/JSON-RPC/SSE，ACP over stdio

本 notebook 跑通两个**真实传输层**实验（不是进程内 mock）：

1. `a2a_wire_demo.py` — 真实 TCP socket 上的 Agent Card 发现、JSON-RPC 2.0、SSE 流；
2. `acp_stdio_demo.py` — 真实子进程 stdio 管道上的双向 JSON-RPC（Zed ACP）。

先看地图：`../images/fig1_protocol_boundaries.png`（协议边界与传输一览）。

## 实验 1：A2A v1 的线上字节

关注四个细节：
- Agent Card 从哪个 well-known 路径取；
- `SendMessage` 的 JSON-RPC envelope 长什么样；
- SSE 帧如何封帧（`data:` 行 + 空行），流如何结束；
- 错误为什么 HTTP 是 200、错误码却在 body 里。

In [1]:

import subprocess, sys
from pathlib import Path

SCRIPTS = Path.cwd().parent / "scripts"

def run(script, *args):
    proc = subprocess.run(
        [sys.executable, str(SCRIPTS / script), *args],
        capture_output=True, text=True, timeout=120,
    )
    print(proc.stdout)
    if proc.returncode != 0:
        print(proc.stderr, file=sys.stderr)
    return proc.returncode

run("a2a_wire_demo.py")

# server listening on http://127.0.0.1:9911 (background thread)

== 1. Discovery: GET /.well-known/agent-card.json ==
  name='Echo Route Planner'  binding=JSONRPC  url=http://127.0.0.1:9911/rpc

== 2. Unary: SendMessage (request -> one Task response) ==
  task.id=task-d1d4bf  state=TASK_STATE_COMPLETED
  artifact: Hello! You said: hello a2a

== 3. Streaming: SendStreamingMessage (SSE frames) ==
  HTTP 200  Content-Type=text/event-stream
  frame: statusUpdate -> {"taskId": "task-da42ad", "contextId": "ctx-5760b7", "status": {"state": "TASK_STATE_WORKING", "time
  frame: artifactUpdate -> {"taskId": "task-da42ad", "contextId": "ctx-5760b7", "artifact": {"artifactId": "art-1", "parts": [{
  frame: artifactUpdate -> {"taskId": "task-da42ad", "contextId": "ctx-5760b7", "artifact": {"artifactId": "art-1", "parts": [{
  frame: statusUpdate -> {"taskId": "task-da42ad", "contextId": "ctx-5760b7", "status": {"state": "TASK_STATE_COMPLETED", "ti
  (server closed the stream = end of run)

== 4. Ge

0

**检查点 1**：
1. `SendStreamingMessage` 的每个 SSE 帧里，`result` 是 `statusUpdate` 还是 `artifactUpdate`？
2. 客户端是怎么知道本次运行结束的？（提示：EOF）
3. 未知 task id 的 `GetTask` 返回的 HTTP 状态和 body 里的错误码各是多少？

## 实验 2：Zed ACP —— stdio 上的双向 JSON-RPC

注意 ACP 与 A2A 的结构差异：
- 传输是**子进程 stdio**，每行一个 JSON 对象；
- `initialize` 握手交换 capability；
- **prompt 未返回时，agent 可以反过来向编辑器发 `fs/read_text_file` 请求**——编辑器握着文件系统。

In [2]:

import subprocess, sys
from pathlib import Path

SCRIPTS = Path.cwd().parent / "scripts"

def run(script, *args):
    proc = subprocess.run(
        [sys.executable, str(SCRIPTS / script), *args],
        capture_output=True, text=True, timeout=120,
    )
    print(proc.stdout)
    if proc.returncode != 0:
        print(proc.stderr, file=sys.stderr)
    return proc.returncode

run("acp_stdio_demo.py", "--verbose")

# spawning agent subprocess: python acp_stdio_demo.py --agent

== 1. initialize (capability handshake) ==
  << {"jsonrpc": "2.0", "id": 1, "result": {"protocolVersion": 1, "agentCapabilities": {"loadSession": false, "promptCapabilities": {"image": false, "audio": false, "embeddedContext": false}, "mcpCapabilities": {"http": false, "sse": false}}, "agentInfo": {"name": "tutorial-agent", "version": "1.0.0"}, "authMethods": []}}
  agent: {'name': 'tutorial-agent', 'version': '1.0.0'}  negotiatedVersion=1

== 2. session/new (editor grants cwd + mcpServers) ==
  << {"jsonrpc": "2.0", "id": 2, "result": {"sessionId": "sess-demo-1"}}
  sessionId=sess-demo-1

== 3. session/prompt (agent streams; editor answers fs/read_text_file) ==
  << {"jsonrpc": "2.0", "method": "session/update", "params": {"sessionId": "sess-demo-1", "update": {"sessionUpdate": "agent_message_chunk", "content": {"type": "text", "text": "I'll inspect the project files first."}}}}
  << {"jsonrpc": "2.0", "method": "session/u

0

**检查点 2**：
1. 在 ACP 里，为什么读取文件是 agent → editor 的请求？放到 A2A 里这对应什么？
2. `session/update` 通知和 JSON-RPC 请求在信封上的区别是什么？（提示：id 字段）

小结：A2A 双方对等、走 HTTP；ACP 客户端拉起 agent、握有资源；两者都以 JSON-RPC 2.0 为语义核心。